In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Validate scope

1. Count to ensure theater had more than 10 showings total.
2. Return a list of all movie titles and verify online they have captioned versions available.
3. Count screenings per film per theater in the week and remove films that had fewer than 4 screenings during the week; these do not fall under the rule.

In [ ]:
## Import .CSV
amc_screenings = pd.read_csv('/Users/jackwalker/Data/2026/CUNY_fall/adv-data-assignment-1/data/import/screenings/raw/amc/amc.csv')
print(amc_screenings)

In [ ]:
## Check for number of screenings per theater
amc_screenings['Theater'].value_counts()

In [ ]:
## Okay, they all have more than 10.

## Now I'm going to store each unique film title in a .CSV file and then
## manually check online to confirm it has a captioned version possible.

open_caption_available = (
    amc_screenings[['Film']]
    .drop_duplicates()
    .sort_values('Film')
    .reset_index(drop=True)
)

open_caption_available['Available?'] = pd.Series(
    pd.NA, index=open_caption_available.index, dtype='boolean'
)

## print(open_caption_available)

## From a manual check online, some are unclear on caption access

captions_missing = [
    'AMC Screen Unseen: November 2',
    'Club Kid with Q&A',
    'I Play Rocky Special Q&A with Cast/Filmmakers',
    'Rammstein - Live in Mexico City',
    "Stray Kids World Tour 'RUN IT' LIVE VIEWING",
    'The Cat in the Hat : Private Theatre Rental',
]

# Set False for those titles, True for all others
open_caption_available['Available?'] = ~open_caption_available['Film'].isin(captions_missing)

print(open_caption_available)

In [ ]:
## Remove the ones we can't confirm from our dataset; this makes our final
## result a minimum number of screenings failing to meet the mark, as 
## opposed to an overshot bc some didn't even have captions as an option.

amc_screenings_filtered = amc_screenings[
    ~amc_screenings['Film'].isin(captions_missing)
].copy()

print('Rows removed:', len(amc_screenings) - len(amc_screenings_filtered))

In [ ]:
## Remove screenings with fewer than four per theater.

## Store counts of films per theater using groupby
screenings_count = amc_screenings_filtered.groupby(['Theater', 'Film']).size()

# Find theater screenings exempt due to frequency criteria
screenings_too_few = screenings_count[screenings_count < 4].index

## Remove those screenings
amc_screenings_filtered = (
    amc_screenings_filtered
    .set_index(['Theater', 'Film'])
    .drop(index=screenings_too_few, errors='ignore')
    .reset_index()
    .copy()
)

print(amc_screenings_filtered)

## 2. Check for proportion of open captioned screenings

1. Identify the number of showings of each film at each theater in the week. Store film name, theater and count in a new dataframe using groupby.
2. Count how many of those showings were open captioned. Store the number in a new column.
3. Calculate the percentage of screenings per film per theater that were open captioned, and store the calculation in a new column.
4. Check if that percent is greater than or equal to the minimum set by state law. Create a new Adhering? column and store a boolean there based on whether or not they're adhering to state law
5. Return a new dataframe just of violations, based on where Adhering? == False
6. Print a count of violations per individual theater.

In [ ]:
## Create our new dataframe

screenings_per_theater = (
    amc_screenings_filtered
    .groupby(['Theater', 'Film'])
    .size()
    .reset_index(name='Total screenings')
)

print(screenings_per_theater)

In [ ]:
## Add a column for open captioned totals
screenings_per_theater['Total captioned'] = (
    amc_screenings_filtered
    .assign(is_oc=amc_screenings_filtered['Open captioned?'].eq('Yes'))
    .groupby(['Theater', 'Film'])['is_oc']
    .sum()
    .values
)

print(screenings_per_theater)

In [ ]:
## Add a column calculating proportion that are open captioned
screenings_per_theater['Captioned proportion'] = (
    screenings_per_theater['Total captioned'] / screenings_per_theater['Total screenings']
)

print(screenings_per_theater)

In [ ]:
## Add a column clarifying whether that meets the state law
screenings_per_theater['Adhering?'] = (
    screenings_per_theater['Total captioned'] >= np.minimum(4, np.ceil(screenings_per_theater['Total screenings'] * 0.25))
)

print(screenings_per_theater)

In [ ]:
# Store violating films/theaters in a new dataframe
violations = screenings_per_theater[screenings_per_theater['Adhering?'] == False].copy()

print(violations)

In [ ]:
amc_violation_count = len(violations)
amc_total_count = len(amc_screenings_filtered)
amc_violation_rate = (amc_violation_count / amc_total_count)

print(f"AMC total relevant screenings: {amc_total_count}.")
print(f"AMC total violations: {amc_violation_count}.")
print(f"From Nov. 2 to 8, 2026, AMC failed to meet open caption laws for {amc_violation_rate * 100:.2f}% of the film titles it screened.")

In [ ]:
## Break down violations per theater

theater_violation_totals = (
    violations.groupby('Theater')
    .size()
    .sort_values(ascending=False)
)

theater_violation_totals_df = theater_violation_totals.reset_index(name='Violation count')

# Filter for totals
theater_screening_totals_df = (
    amc_screenings_filtered.groupby('Theater')
    .size()
    .reset_index(name='Total screenings')
)

#Filter for violations
theater_violation_totals_df = (
    theater_violation_totals_df
    .merge(theater_screening_totals_df, on='Theater', how='left')
    .sort_values('Violation count', ascending=False)
    .reset_index(drop=True)
)

#Calculate rate of violations
theater_violation_totals_df = (
    theater_violation_totals_df
    .assign(**{
        'Violation rate (%)': (
            theater_violation_totals_df['Violation count']
            / theater_violation_totals_df['Total screenings'] * 100
        ).round(2)
    })
)

print(theater_violation_totals_df.sort_values('Violation rate (%)', ascending=False))

In [ ]:
# Export selected dataframes
base_export_dir = Path('/Users/jackwalker/Data/2026/CUNY_fall/adv-data-assignment-1/data/export/screenings/amc')
violations_by_total_dir = base_export_dir / 'violations-by-total'
screenings_filtered_dir = base_export_dir / 'screenings-filtered-dataset'

exports = {
    screenings_filtered_dir / 'amc_screenings_relevant_to_law.csv': amc_screenings_filtered,
    violations_by_total_dir / 'screenings_per_theater.csv': screenings_per_theater,
    violations_by_total_dir / 'theater_violation_totals.csv': theater_violation_totals_df,
}

for filepath, df in exports.items():
    df.to_csv(filepath, index=False)

print('Exported files:')
for filepath in exports:
    print('-', filepath)